In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random
from urllib.parse import quote, urlencode
import re
from datetime import datetime
import os
from dotenv import load_dotenv
import json
import logging
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import openpyxl
from openpyxl.styles import Font, Alignment
import warnings
warnings.filterwarnings('ignore')

class NaverImageScraper:
    def __init__(self, search_year=2024, search_month=6, max_total_images=10000):
        """
        네이버 이미지 검색 스크래퍼 초기화
        
        Args:
            search_year (int): 검색할 년도
            search_month (int): 검색할 월
            max_total_images (int): 최대 수집 이미지 수
        """
        load_dotenv()  # .env 파일 로드
        
        self.search_year = search_year
        self.search_month = search_month
        self.max_total_images = max_total_images
        self.current_total_count = 0
        
        # 결과 저장용 리스트
        self.results = []
        
        # 로깅 설정
        logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
        self.logger = logging.getLogger(__name__)
        
        # 셀레니움 드라이버 설정
        self.setup_driver()
        
        print(f"네이버 이미지 스크래퍼 초기화 완료")
        print(f"검색 기간: {search_year}년 {search_month}월")
        print(f"최대 수집 이미지: {max_total_images:,}개")
    
    def setup_driver(self):
        """셀레니움 웹드라이버 설정"""
        chrome_options = Options()
        # 헤드리스 모드는 디버깅을 위해 일시적으로 비활성화
        # chrome_options.add_argument('--headless')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-dev-shm-usage')
        chrome_options.add_argument('--disable-gpu')
        chrome_options.add_argument('--window-size=1920,1080')
        chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_experimental_option('useAutomationExtension', False)
        
        try:
            self.driver = webdriver.Chrome(options=chrome_options)
            self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            self.wait = WebDriverWait(self.driver, 20)
        except Exception as e:
            self.logger.error(f"웹드라이버 설정 실패: {e}")
            raise
    
    def load_menu_data(self, csv_file_path):
        """CSV 파일에서 메뉴 데이터 로드"""
        try:
            df = pd.read_csv(csv_file_path, encoding='utf-8')
            self.logger.info(f"CSV 파일 로드 완료: {len(df)}개 행")
            
            # 상세메뉴를 개별 항목으로 분리
            menu_items = []
            for _, row in df.iterrows():
                detail_menus = [menu.strip() for menu in str(row['상세메뉴']).split(',') if menu.strip()]
                for detail_menu in detail_menus:
                    menu_items.append({
                        '대분류': row['대분류'],
                        '중분류': row['중분류'],
                        '소분류': row['소분류'],
                        '상세메뉴': detail_menu,
                        '시각적특징': row['시각적특징']
                    })
            
            self.logger.info(f"총 {len(menu_items)}개의 개별 메뉴 항목 생성")
            return menu_items
            
        except Exception as e:
            self.logger.error(f"CSV 파일 로드 실패: {e}")
            raise
    
    def search_naver_images(self, keyword, apply_date_filter=True):
        """네이버 이미지 검색 수행"""
        try:
            # 네이버 이미지 검색 페이지로 이동
            search_url = f"https://search.naver.com/search.naver?where=image&query={quote(keyword)}"
            print(f"검색 URL: {search_url}")
            
            self.driver.get(search_url)
            time.sleep(random.uniform(3, 5))
            
            # 날짜 필터 적용
            if apply_date_filter:
                success = self.apply_date_filter()
                if not success:
                    print(f"날짜 필터 적용 실패, 전체 기간으로 검색합니다.")
            
            # 이미지 수집
            images = self.collect_images_with_scroll()
            
            self.logger.info(f"'{keyword}' 검색 완료: {len(images)}개 이미지 수집")
            return images
            
        except Exception as e:
            self.logger.error(f"이미지 검색 실패 ({keyword}): {e}")
            return []
    
    def apply_date_filter(self):
        """날짜 필터 적용"""
        try:
            # 검색 옵션 버튼 찾기 (더보기, 옵션 등)
            option_buttons = [
                "//button[contains(text(), '옵션')]",
                "//a[contains(@class, 'option')]",
                "//button[contains(@class, 'filter')]",
                "//*[contains(text(), '기간')]",
                "//button[contains(text(), '더보기')]"
            ]
            
            option_clicked = False
            for xpath in option_buttons:
                try:
                    option_btn = self.driver.find_element(By.XPATH, xpath)
                    if option_btn.is_displayed():
                        option_btn.click()
                        option_clicked = True
                        time.sleep(2)
                        break
                except:
                    continue
            
            if not option_clicked:
                # URL 파라미터로 날짜 필터 적용
                start_date = f"{self.search_year}{self.search_month:02d}01"
                
                # 해당 월의 마지막 날 계산
                if self.search_month in [1, 3, 5, 7, 8, 10, 12]:
                    last_day = 31
                elif self.search_month in [4, 6, 9, 11]:
                    last_day = 30
                else:  # 2월
                    if self.search_year % 4 == 0:
                        last_day = 29
                    else:
                        last_day = 28
                
                end_date = f"{self.search_year}{self.search_month:02d}{last_day:02d}"
                
                current_url = self.driver.current_url
                if "?" in current_url:
                    filtered_url = f"{current_url}&pd=3&ds={start_date}&de={end_date}"
                else:
                    filtered_url = f"{current_url}?pd=3&ds={start_date}&de={end_date}"
                
                self.driver.get(filtered_url)
                time.sleep(3)
                return True
            
            # 기간 설정 찾기
            date_elements = [
                "//button[contains(text(), '기간')]",
                "//*[contains(text(), '1개월')]",
                "//*[contains(text(), '직접입력')]"
            ]
            
            for xpath in date_elements:
                try:
                    date_btn = self.driver.find_element(By.XPATH, xpath)
                    if date_btn.is_displayed():
                        date_btn.click()
                        time.sleep(2)
                        
                        # 직접입력 또는 특정 기간 설정
                        self.set_custom_date_range()
                        return True
                except:
                    continue
            
            return False
            
        except Exception as e:
            self.logger.error(f"날짜 필터 적용 실패: {e}")
            return False
    
    def set_custom_date_range(self):
        """사용자 정의 날짜 범위 설정"""
        try:
            # 시작일 입력
            start_inputs = self.driver.find_elements(By.CSS_SELECTOR, "input[type='text']")
            for inp in start_inputs:
                placeholder = inp.get_attribute('placeholder') or ''
                if '시작' in placeholder or 'from' in placeholder.lower():
                    inp.clear()
                    inp.send_keys(f"{self.search_year}.{self.search_month:02d}.01")
                    break
            
            # 종료일 입력
            if self.search_month in [1, 3, 5, 7, 8, 10, 12]:
                last_day = 31
            elif self.search_month in [4, 6, 9, 11]:
                last_day = 30
            else:
                last_day = 29 if self.search_year % 4 == 0 else 28
            
            end_inputs = self.driver.find_elements(By.CSS_SELECTOR, "input[type='text']")
            for inp in end_inputs:
                placeholder = inp.get_attribute('placeholder') or ''
                if '종료' in placeholder or 'to' in placeholder.lower():
                    inp.clear()
                    inp.send_keys(f"{self.search_year}.{self.search_month:02d}.{last_day:02d}")
                    break
            
            # 적용 버튼 클릭
            apply_btns = [
                "//button[contains(text(), '적용')]",
                "//button[contains(text(), '확인')]",
                "//input[@type='submit']"
            ]
            
            for xpath in apply_btns:
                try:
                    apply_btn = self.driver.find_element(By.XPATH, xpath)
                    if apply_btn.is_displayed():
                        apply_btn.click()
                        time.sleep(3)
                        break
                except:
                    continue
                    
        except Exception as e:
            self.logger.error(f"날짜 범위 설정 실패: {e}")
    
    def collect_images_with_scroll(self, max_images=1000):
        """스크롤하면서 이미지 수집"""
        images = []
        last_height = self.driver.execute_script("return document.body.scrollHeight")
        scroll_count = 0
        max_scrolls = 20
        
        try:
            while len(images) < max_images and scroll_count < max_scrolls:
                # 현재 보이는 이미지들 수집
                current_images = self.extract_visible_images()
                
                # 새로운 이미지들만 추가
                for img in current_images:
                    if img not in images and len(images) < max_images:
                        images.append(img)
                
                # 페이지 아래로 스크롤
                self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(random.uniform(2, 4))
                
                # 새로운 컨텐츠가 로드될 때까지 대기
                try:
                    self.wait.until(lambda driver: driver.execute_script("return document.body.scrollHeight") > last_height)
                except:
                    # 더 이상 로드할 컨텐츠가 없는 경우
                    break
                
                new_height = self.driver.execute_script("return document.body.scrollHeight")
                if new_height == last_height:
                    # 더 보기 버튼 찾기
                    try:
                        more_btn = self.driver.find_element(By.XPATH, "//button[contains(text(), '더보기') or contains(text(), '더 보기')]")
                        if more_btn.is_displayed():
                            more_btn.click()
                            time.sleep(3)
                        else:
                            break
                    except:
                        break
                
                last_height = new_height
                scroll_count += 1
                
                if scroll_count % 5 == 0:
                    print(f"  스크롤 {scroll_count}회, 현재 {len(images)}개 이미지 수집")
            
            return images[:max_images]
            
        except Exception as e:
            self.logger.error(f"이미지 수집 중 오류: {e}")
            return images
    
    def extract_visible_images(self):
        """현재 화면에 보이는 이미지들 추출"""
        images = []
        
        try:
            # 다양한 이미지 선택자 시도
            image_selectors = [
                "img[src*='blogfiles.naver.net']",
                "img[src*='pstatic.net']",
                "img[data-src*='blogfiles.naver.net']",
                "img[data-src*='pstatic.net']",
                "img._img",
                ".thumb img",
                ".photo img",
                "a[href*='blog.naver.com'] img",
                "a[href*='cafe.naver.com'] img"
            ]
            
            for selector in image_selectors:
                try:
                    img_elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                    
                    for img in img_elements:
                        try:
                            img_data = self.extract_image_data(img)
                            if img_data and img_data not in images:
                                images.append(img_data)
                        except:
                            continue
                            
                except:
                    continue
            
            # 중복 제거
            unique_images = []
            seen_urls = set()
            
            for img in images:
                if img['image_url'] not in seen_urls:
                    unique_images.append(img)
                    seen_urls.add(img['image_url'])
            
            return unique_images
            
        except Exception as e:
            self.logger.error(f"이미지 추출 실패: {e}")
            return []
    
    def extract_image_data(self, img_element):
        """개별 이미지 데이터 추출"""
        try:
            # 이미지 URL 추출
            img_url = (img_element.get_attribute('src') or 
                      img_element.get_attribute('data-src') or 
                      img_element.get_attribute('data-lazy-src'))
            
            if not img_url or 'data:image' in img_url:
                return None
            
            # 이미지 제목/설명 추출
            img_title = (img_element.get_attribute('alt') or 
                        img_element.get_attribute('title') or '')
            
            # 상위 요소에서 링크 URL 찾기
            source_url = ''
            try:
                parent_link = img_element.find_element(By.XPATH, "./ancestor::a[1]")
                source_url = parent_link.get_attribute('href') or ''
            except:
                pass
            
            # 추가 텍스트 정보 수집 (주변 텍스트)
            additional_text = ''
            try:
                # 상위 div에서 텍스트 찾기
                parent_div = img_element.find_element(By.XPATH, "./ancestor::div[1]")
                text_elements = parent_div.find_elements(By.TAG_NAME, "span")
                text_elements.extend(parent_div.find_elements(By.TAG_NAME, "p"))
                text_elements.extend(parent_div.find_elements(By.TAG_NAME, "div"))
                
                texts = []
                for elem in text_elements:
                    text = elem.text.strip()
                    if text and len(text) < 100:
                        texts.append(text)
                
                additional_text = ' '.join(texts[:3])  # 최대 3개까지
            except:
                pass
            
            return {
                'image_url': img_url,
                'title': img_title,
                'source_url': source_url,
                'additional_text': additional_text,
                'collected_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            }
            
        except Exception as e:
            return None
    
    def filter_images_by_menu(self, images, menu_item):
        """메뉴 키워드로 이미지 필터링"""
        menu_name = menu_item['상세메뉴']
        filtered_images = []
        
        # 키워드 매칭
        for img in images:
            title = (img.get('title', '') + ' ' + img.get('additional_text', '')).lower()
            menu_lower = menu_name.lower()
            
            # 메뉴명이 텍스트에 포함되어 있는지 확인
            if (menu_lower in title or 
                any(word in title for word in menu_lower.split()) or
                any(menu_word in title for menu_word in menu_name.split())):
                
                img_copy = img.copy()
                img_copy.update({
                    '대분류': menu_item['대분류'],
                    '중분류': menu_item['중분류'],
                    '소분류': menu_item['소분류'],
                    '상세메뉴': menu_name,
                    '시각적특징': menu_item['시각적특징'],
                    '업로드시기': f"{self.search_year}-{self.search_month:02d}",
                    '검색키워드': "단품음식이미지"
                })
                filtered_images.append(img_copy)
        
        return filtered_images
    
    def process_all_menus(self, csv_file_path):
        """모든 메뉴에 대해 이미지 검색 및 수집"""
        # 메뉴 데이터 로드
        menu_items = self.load_menu_data(csv_file_path)
        
        # 메인 키워드로 기본 검색 먼저 수행
        print("메인 키워드 '단품음식이미지' 검색 중...")
        main_images = self.search_naver_images("단품음식이미지", apply_date_filter=True)
        
        if not main_images:
            print("메인 키워드에서 이미지를 찾을 수 없습니다. 개별 메뉴 검색을 시도합니다.")
            # 메인 검색 실패시 개별 메뉴로 검색
            return self.process_individual_menus(menu_items)
        
        print(f"메인 검색에서 {len(main_images)}개 이미지 발견")
        
        # 각 메뉴별로 이미지 필터링
        processed_count = 0
        
        for i, menu_item in enumerate(menu_items):
            if self.current_total_count >= self.max_total_images:
                print(f"\n최대 수집량({self.max_total_images:,}개)에 도달하여 중단합니다.")
                break
            
            menu_name = menu_item['상세메뉴']
            print(f"\n[{i+1}/{len(menu_items)}] '{menu_name}' 필터링 중...")
            
            # 메뉴별 이미지 필터링
            filtered_images = self.filter_images_by_menu(main_images, menu_item)
            
            if filtered_images:
                # 최대 수집량 체크
                remaining_quota = self.max_total_images - self.current_total_count
                images_to_add = filtered_images[:remaining_quota]
                
                self.results.extend(images_to_add)
                self.current_total_count += len(images_to_add)
                
                print(f"  → {len(images_to_add)}개 이미지 수집 (총 {self.current_total_count:,}개)")
            else:
                print(f"  → 매칭되는 이미지 없음")
            
            processed_count += 1
            
            # 진행률 표시
            if processed_count % 20 == 0:
                progress = (processed_count / len(menu_items)) * 100
                print(f"\n진행률: {progress:.1f}% ({processed_count}/{len(menu_items)})")
        
        print(f"\n전체 처리 완료!")
        print(f"총 수집 이미지: {self.current_total_count:,}개")
    
    def process_individual_menus(self, menu_items):
        """개별 메뉴별로 직접 검색"""
        print("개별 메뉴 검색 모드로 전환...")
        
        processed_count = 0
        
        for i, menu_item in enumerate(menu_items):
            if self.current_total_count >= self.max_total_images:
                print(f"\n최대 수집량({self.max_total_images:,}개)에 도달하여 중단합니다.")
                break
            
            menu_name = menu_item['상세메뉴']
            search_keyword = f"단품음식이미지 {menu_name}"
            
            print(f"\n[{i+1}/{len(menu_items)}] '{search_keyword}' 검색 중...")
            
            # 개별 검색 수행
            images = self.search_naver_images(search_keyword, apply_date_filter=True)
            
            if images:
                # 결과에 메뉴 정보 추가
                for img in images:
                    img.update({
                        '대분류': menu_item['대분류'],
                        '중분류': menu_item['중분류'],
                        '소분류': menu_item['소분류'],
                        '상세메뉴': menu_name,
                        '시각적특징': menu_item['시각적특징'],
                        '업로드시기': f"{self.search_year}-{self.search_month:02d}",
                        '검색키워드': search_keyword
                    })
                
                # 최대 수집량 체크
                remaining_quota = self.max_total_images - self.current_total_count
                images_to_add = images[:remaining_quota]
                
                self.results.extend(images_to_add)
                self.current_total_count += len(images_to_add)
                
                print(f"  → {len(images_to_add)}개 이미지 수집 (총 {self.current_total_count:,}개)")
            else:
                print(f"  → 검색 결과 없음")
            
            processed_count += 1
            
            # 요청 간격
            time.sleep(random.uniform(3, 6))
            
            # 진행률 표시
            if processed_count % 10 == 0:
                progress = (processed_count / len(menu_items)) * 100
                print(f"\n진행률: {progress:.1f}% ({processed_count}/{len(menu_items)})")
        
        print(f"\n개별 검색 완료!")
        print(f"총 수집 이미지: {self.current_total_count:,}개")
    
    def save_to_excel(self, output_file="naver_image_search_results.xlsx"):
        """결과를 엑셀 파일로 저장"""
        if not self.results:
            print("저장할 결과가 없습니다.")
            return
        
        try:
            # 데이터프레임 생성
            df = pd.DataFrame(self.results)
            
            # 컬럼 순서 정리
            columns_order = [
                '상세메뉴', '대분류', '중분류', '소분류', '시각적특징',
                'image_url', 'title', 'additional_text', 'source_url',
                '업로드시기', '검색키워드', 'collected_at'
            ]
            
            # 존재하는 컬럼만 선택
            available_columns = [col for col in columns_order if col in df.columns]
            df = df[available_columns]
            
            # 중복 제거 (같은 이미지 URL)
            df = df.drop_duplicates(subset=['image_url'], keep='first')
            
            # 엑셀 파일로 저장
            with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
                df.to_excel(writer, sheet_name='검색결과', index=False)
                
                # 워크시트 스타일링
                worksheet = writer.sheets['검색결과']
                
                # 헤더 스타일
                header_font = Font(bold=True)
                header_alignment = Alignment(horizontal='center')
                
                for cell in worksheet[1]:
                    cell.font = header_font
                    cell.alignment = header_alignment
                
                # 컬럼 너비 조정
                for column in worksheet.columns:
                    max_length = 0
                    column_letter = column[0].column_letter
                    
                    for cell in column:
                        try:
                            if len(str(cell.value)) > max_length:
                                max_length = len(str(cell.value))
                        except:
                            pass
                    
                    adjusted_width = min(max_length + 2, 50)
                    worksheet.column_dimensions[column_letter].width = adjusted_width
            
            # 통계 정보 저장
            self.save_statistics(output_file)
            
            print(f"\n결과 저장 완료: {output_file}")
            print(f"총 {len(df)}개 이미지 정보 저장 (중복 제거 후)")
            
        except Exception as e:
            self.logger.error(f"엑셀 저장 실패: {e}")
            
            # 백업으로 CSV 저장
            try:
                df.to_csv(output_file.replace('.xlsx', '.csv'), encoding='utf-8-sig', index=False)
                print(f"백업 CSV 파일로 저장: {output_file.replace('.xlsx', '.csv')}")
            except Exception as csv_error:
                self.logger.error(f"CSV 백업 저장도 실패: {csv_error}")
    
    def save_statistics(self, excel_file):
        """통계 정보 저장"""
        try:
            df = pd.DataFrame(self.results)
            
            # 메뉴별 통계
            menu_stats = df.groupby('상세메뉴').size().reset_index(name='이미지수')
            menu_stats = menu_stats.sort_values('이미지수', ascending=False)
            
            # 분류별 통계
            category_stats = df.groupby(['대분류', '중분류']).size().reset_index(name='이미지수')
            category_stats = category_stats.sort_values('이미지수', ascending=False)
            
            # 검색키워드별 통계
            keyword_stats = df.groupby('검색키워드').size().reset_index(name='이미지수')
            keyword_stats = keyword_stats.sort_values('이미지수', ascending=False)
            
            with pd.ExcelWriter(excel_file, mode='a', engine='openpyxl') as writer:
                menu_stats.to_excel(writer, sheet_name='메뉴별통계', index=False)
                category_stats.to_excel(writer, sheet_name='분류별통계', index=False)
                keyword_stats.to_excel(writer, sheet_name='키워드별통계', index=False)
                
                # 전체 요약
                summary_data = {
                    '항목': [
                        '전체 수집 이미지',
                        '고유 메뉴 수',
                        '검색 기간',
                        '수집 일시',
                        '평균 메뉴당 이미지수'
                    ],
                    '값': [
                        f"{len(df):,}개",
                        f"{df['상세메뉴'].nunique()}개",
                        f"{self.search_year}년 {self.search_month}월",
                        datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                        f"{len(df) / df['상세메뉴'].nunique():.1f}개"
                    ]
                }
                
                summary_df = pd.DataFrame(summary_data)
                summary_df.to_excel(writer, sheet_name='수집요약', index=False)
            
        except Exception as e:
            self.logger.error(f"통계 저장 실패: {e}")
    
    def close(self):
        """리소스 정리"""
        if hasattr(self, 'driver'):
            self.driver.quit()
        print("스크래퍼 종료")


def main():
    """메인 실행 함수"""
    # 설정값
    CSV_FILE_PATH = "식당대12중53소132상세메뉴379분류.csv"
    SEARCH_YEAR = 2024
    SEARCH_MONTH = 6
    MAX_IMAGES = 10000
    OUTPUT_FILE = f"naver_image_results_{SEARCH_YEAR}{SEARCH_MONTH:02d}.xlsx"
    
    scraper = None
    
    try:
        print("=" * 60)
        print("네이버 이미지 검색 자동화 시작")
        print("=" * 60)
        
        # 스크래퍼 초기화
        scraper = NaverImageScraper(
            search_year=SEARCH_YEAR,
            search_month=SEARCH_MONTH,
            max_total_images=MAX_IMAGES
        )
        
        # CSV 파일 확인
        if not os.path.exists(CSV_FILE_PATH):
            print(f"오류: CSV 파일을 찾을 수 없습니다 - {CSV_FILE_PATH}")
            return
        
        # 검색 실행
        scraper.process_all_menus(CSV_FILE_PATH)
        
        # 결과 저장
        scraper.save_to_excel(OUTPUT_FILE)
        
        print("\n" + "=" * 60)
        print("작업 완료!")
        print("=" * 60)
        
    except KeyboardInterrupt:
        print("\n사용자에 의해 중단되었습니다.")
        
    except Exception as e:
        print(f"\n오류 발생: {e}")
        import traceback
        traceback.print_exc()
        
    finally:
        if scraper:
            scraper.close()


if __name__ == "__main__":
    main()

네이버 이미지 검색 자동화 시작


2025-07-14 23:41:00,645 - INFO - CSV 파일 로드 완료: 138개 행
2025-07-14 23:41:00,658 - INFO - 총 381개의 개별 메뉴 항목 생성


네이버 이미지 스크래퍼 초기화 완료
검색 기간: 2024년 6월
최대 수집 이미지: 10,000개
메인 키워드 '단품음식이미지' 검색 중...
검색 URL: https://search.naver.com/search.naver?where=image&query=%EB%8B%A8%ED%92%88%EC%9D%8C%EC%8B%9D%EC%9D%B4%EB%AF%B8%EC%A7%80
  스크롤 5회, 현재 705개 이미지 수집


2025-07-14 23:50:01,451 - ERROR - 이미지 수집 중 오류: Message: invalid session id
Stacktrace:
	GetHandleVerifier [0x0x7ff64fe56f75+76917]
	GetHandleVerifier [0x0x7ff64fe56fd0+77008]
	(No symbol) [0x0x7ff64fc09c1c]
	(No symbol) [0x0x7ff64fc5055f]
	(No symbol) [0x0x7ff64fc88332]
	(No symbol) [0x0x7ff64fc82e53]
	(No symbol) [0x0x7ff64fc81f19]
	(No symbol) [0x0x7ff64fbd4b05]
	GetHandleVerifier [0x0x7ff65012d2ad+3051437]
	GetHandleVerifier [0x0x7ff650127903+3028483]
	GetHandleVerifier [0x0x7ff65014589d+3151261]
	GetHandleVerifier [0x0x7ff64fe7183e+185662]
	GetHandleVerifier [0x0x7ff64fe796ff+218111]
	(No symbol) [0x0x7ff64fbd3b00]
	GetHandleVerifier [0x0x7ff650245f18+4201496]
	BaseThreadInitThunk [0x0x7ff9108ae8d7+23]
	RtlUserThreadStart [0x0x7ff91255c34c+44]

2025-07-14 23:50:01,451 - INFO - '단품음식이미지' 검색 완료: 1000개 이미지 수집


메인 검색에서 1000개 이미지 발견

[1/381] '제육볶음' 필터링 중...
  → 매칭되는 이미지 없음

[2/381] '매운제육볶음' 필터링 중...
  → 매칭되는 이미지 없음

[3/381] '두부제육볶음' 필터링 중...
  → 매칭되는 이미지 없음

[4/381] '된장찌개' 필터링 중...
  → 매칭되는 이미지 없음

[5/381] '김치찌개' 필터링 중...
  → 매칭되는 이미지 없음

[6/381] '청국장찌개' 필터링 중...
  → 매칭되는 이미지 없음

[7/381] '콩나물무침' 필터링 중...
  → 매칭되는 이미지 없음

[8/381] '시금치나물' 필터링 중...
  → 매칭되는 이미지 없음

[9/381] '도라지무침' 필터링 중...
  → 매칭되는 이미지 없음

[10/381] '계란말이' 필터링 중...
  → 매칭되는 이미지 없음

[11/381] '계란찜' 필터링 중...
  → 매칭되는 이미지 없음

[12/381] '스크램블에그' 필터링 중...
  → 매칭되는 이미지 없음

[13/381] '미역국' 필터링 중...
  → 매칭되는 이미지 없음

[14/381] '무국' 필터링 중...
  → 매칭되는 이미지 없음

[15/381] '콩나물국' 필터링 중...
  → 매칭되는 이미지 없음

[16/381] '부대찌개' 필터링 중...
  → 매칭되는 이미지 없음

[17/381] '햄부대찌개' 필터링 중...
  → 매칭되는 이미지 없음

[18/381] '치즈부대찌개' 필터링 중...
  → 매칭되는 이미지 없음

[19/381] '감자탕' 필터링 중...
  → 매칭되는 이미지 없음

[20/381] '뼈해장국' 필터링 중...
  → 매칭되는 이미지 없음

진행률: 5.2% (20/381)

[21/381] '등뼈찜' 필터링 중...
  → 매칭되는 이미지 없음

[22/381] '순두부찌개' 필터링 중...
  → 매칭되는 이미지 없음

[23/381] '해물순두부찌개' 필터링 중...
  → 매칭되

  → 매칭되는 이미지 없음

[259/381] '스크램블' 필터링 중...
  → 매칭되는 이미지 없음

[260/381] '오믈렛' 필터링 중...
  → 매칭되는 이미지 없음

진행률: 68.2% (260/381)

[261/381] '팬케이크' 필터링 중...
  → 매칭되는 이미지 없음

[262/381] '프렌치토스트' 필터링 중...
  → 매칭되는 이미지 없음

[263/381] '시저샐러드' 필터링 중...
  → 매칭되는 이미지 없음

[264/381] '코브샐러드' 필터링 중...
  → 매칭되는 이미지 없음

[265/381] '양꼬치' 필터링 중...
  → 매칭되는 이미지 없음

[266/381] '꼬치구이' 필터링 중...
  → 매칭되는 이미지 없음

[267/381] '바베큐' 필터링 중...
  → 매칭되는 이미지 없음

[268/381] '딤섬' 필터링 중...
  → 매칭되는 이미지 없음

[269/381] '찜 요리' 필터링 중...
  → 매칭되는 이미지 없음

[270/381] '쌀국수' 필터링 중...
  → 매칭되는 이미지 없음

[271/381] '분보' 필터링 중...
  → 매칭되는 이미지 없음

[272/381] '분짜' 필터링 중...
  → 매칭되는 이미지 없음

[273/381] '월남쌈' 필터링 중...
  → 매칭되는 이미지 없음

[274/381] '고이쿤' 필터링 중...
  → 매칭되는 이미지 없음

[275/381] '반미' 필터링 중...
  → 매칭되는 이미지 없음

[276/381] '바게트' 필터링 중...
  → 매칭되는 이미지 없음

[277/381] '팟타이' 필터링 중...
  → 매칭되는 이미지 없음

[278/381] '태국볶음면' 필터링 중...
  → 매칭되는 이미지 없음

[279/381] '그린커리' 필터링 중...
  → 매칭되는 이미지 없음

[280/381] '레드커리' 필터링 중...
  → 매칭되는 이미지 없음

진행률: 73.5% (280/381)

[281